In [ ]:
import torch
import numpy as np
import re
import gc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from nnsight import LanguageModel
import nnsight
import importlib

import plot_helpers
importlib.reload(plot_helpers)
from plot_helpers import (
    setup_plotting_style,
    style_ax,
    get_step_formatter,
    save_figure,
    PAPER_WIDTH_IN,
    PAPER_HEIGHT_IN,
)

setup_plotting_style(style="paper")

EXPERIMENT_DIR = "/p/project1/westai0065/hierarchical-latent-structures/experiments/pcfg_large/9ab00aa2-6728-4d27-a894-c5bb1aad6f5d/checkpoints"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16

In [ ]:

def generate_fv_data(batch_size=16):
    source_prompts = []
    target_prompts = []
    source_targets = []
    target_targets = []

    gen_rng = torch.Generator().manual_seed(42)

    for _ in range(batch_size):
        s1 = torch.randint(0, 300, (1,), generator=gen_rng).item()
        v1 = torch.randint(300, 600, (1,), generator=gen_rng).item()
        o1 = torch.randint(600, 900, (1,), generator=gen_rng).item()
        
        s2 = torch.randint(0, 300, (1,), generator=gen_rng).item()
        v2 = torch.randint(300, 600, (1,), generator=gen_rng).item()
        o2 = torch.randint(600, 900, (1,), generator=gen_rng).item()
        
        src_ids = [s1, v1, o1, 950, s1, v1]
        tgt_ids = [s2, v2, o2, 950, s2, v2]
        
        source_prompts.append(torch.tensor(src_ids))
        target_prompts.append(torch.tensor(tgt_ids))
        
        source_targets.append(o1)
        target_targets.append(o2)

    return (
        torch.stack(source_prompts),
        torch.stack(target_prompts),
        torch.tensor(source_targets),
        torch.tensor(target_targets)
    )


def get_sorted_checkpoints(experiment_dir):
    Analyzes function vectors across all checkpoints in an experiment directory.
    
    Returns:
        history_fv_scores: dictionary mapping step_number -> [layer_0_score, layer_1_score, ...]
        steps_processed: list of step numbers processed

In [ ]:

if not history_fv_scores:
    print("No data collected.")
else:
    print("\nPlotting Heatmap...")
    
    matrix = np.array([history_fv_scores[s] for s in sorted(history_fv_scores.keys())])
    
    data = matrix.T
    steps = sorted(history_fv_scores.keys())
    layer_labels = [f"{i}" for i in range(data.shape[0])]
    
    fig, ax = plt.subplots(figsize=(PAPER_WIDTH_IN, PAPER_HEIGHT_IN))
    
    im = ax.imshow(
        data,
        aspect='auto',
        cmap='crest',
        interpolation='nearest',
        origin='lower'
    )
    
    ax.set_xlabel('Training Step', fontsize=11, color='black')
    ax.set_ylabel('Layer', fontsize=11, color='black')
    ax.set_title("Function vector improvement", fontsize=11, pad=8, color='black')
    ax.tick_params(colors='black', labelsize=8)
    
    n_ticks = 4
    n_steps = len(steps)
    step_positions = np.linspace(0, n_steps - 1, n_ticks, dtype=int)
    ax.set_xticks(step_positions)
    ax.set_xticklabels([steps[pos] for pos in step_positions])
    ax.xaxis.set_major_formatter(get_step_formatter())
    
    ax.set_yticks(np.arange(len(layer_labels)))
    ax.set_yticklabels(layer_labels)
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=8, colors='black')
    for spine in cbar.ax.spines.values():
        spine.set_color('black')
    
    style_ax(ax, xlabel='Training Step', ylabel='Layer', style='paper')
    ax.set_title("Function vector improvement", fontsize=11, pad=8, color='black')
    
    for spine in ax.spines.values():
        spine.set_color('black')
    
    plt.tight_layout()
    save_figure(fig, "fv_attention_scores_heatmap.pdf")
    plt.show()

No data collected.
